# ⚙️ Qdrant Vector DB - Native Payload Boosting & Recency Decay Evaluation

This notebook demonstrates how to utilize **Qdrant's native query engine modifiers** to perform score boosting and recency decay directly inside the database server.

### 🌟 Why use Qdrant?
1. **Native Score Modifiers**: Allows you to apply metadata boosts (e.g., flat score addition if `data_type == 'explicit'`) directly in the database engine.
2. **Native Time Decay**: Uses mathematical decay expressions (Exponential, Linear, or Gaussian) to decay vector similarity scores based on chronological fields directly inside the query pipeline.
3. **Performance**: Eliminates custom Python post-processing loops for decay/boosting, keeping all operations inside optimized database memory.

--- 
## 🤖 Does Qdrant Provide Embedding and Reranking Natively?

- **Embedding Models**: The Qdrant *database server* does not generate vector embeddings natively. However, Qdrant provides a built-in CPU-optimized Python library called **`FastEmbed`** (`qdrant-client[fastembed]`). This allows you to generate embeddings in your Python code using pre-built models (like `BAAI/bge-small-en-v1.5` or `nomic-ai/nomic-embed-text-v1.5`) without needing a heavy PyTorch/SentenceTransformers installation.
- **Reranking Models**: Similar to embeddings, the database server itself does not run Cross-Encoder models natively. However, you can run rerankers on the client side using **`FastEmbed`'s `TextReranker`** or the Cohere/Voyage AI APIs right after fetching candidate results from Qdrant.

In [12]:
# 1. Install Qdrant Client
!pip install -q qdrant-client


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
# 2. Imports and Environment Setup
import time
import sys
import os

# Add parent directory to sys.path to resolve database and test imports correctly
notebook_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(notebook_dir, ".."))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct,
    Prefetch,
    FormulaQuery,
    SumExpression,
    MultExpression,
    FieldCondition,
    MatchValue,
    ExpDecayExpression,
    DecayParamsExpression
)

# Import actual evaluation dataset and database engine config
from test_semantic_search import TEST_MEMORIES, EVAL_QUERIES, DISTRACTOR_QUERIES
import database

# Load the same 768-dimension Nomic Embedding model used in the main app
model = database.get_transformer_model()
print(f"Loaded {len(TEST_MEMORIES)} ground-truth memories.")
print("Embedding Model Dimension:", model.get_sentence_embedding_dimension())

Loaded 50 ground-truth memories.
Embedding Model Dimension: 768


C:\Users\rishabh.nahar\AppData\Local\Temp\ipykernel_25388\2135509236.py:36: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding Model Dimension:", model.get_sentence_embedding_dimension())


In [14]:
# 3. Connect to In-Memory Qdrant Database (For Test Evaluation)
eval_client = QdrantClient(":memory:")
EVAL_COLLECTION_NAME = "agent_memory_nomic"

# Re-create collection with 768 dimensions configured config (matching Nomic)
if eval_client.collection_exists(collection_name=EVAL_COLLECTION_NAME):
    eval_client.delete_collection(collection_name=EVAL_COLLECTION_NAME)
eval_client.create_collection(
    collection_name=EVAL_COLLECTION_NAME,
    vectors_config=VectorParams(size=768, distance=Distance.COSINE)
)
print(f"Collection '{EVAL_COLLECTION_NAME}' is ready for ingestion.")

Collection 'agent_memory_nomic' is ready for ingestion.


In [15]:
# 4. Seed Collection with the actual 50 memories and dynamic timestamps
points = []
current_epoch_time = int(time.time())

print("Generating embeddings and preparing records...")
for idx, mem in enumerate(TEST_MEMORIES):
    # Prepend Nomic's required document prefix (matching database.py)
    combined_text = f"search_document: {mem['q']} {mem['r']}"
    vector = model.encode(combined_text).tolist()
    
    # Distribute timestamps dynamically between 0 and 720 hours ago
    hours_ago = (idx * 15) % 720
    timestamp = current_epoch_time - (hours_ago * 3600)
    
    points.append(
        PointStruct(
            id=mem["id"],
            vector=vector,
            payload={
                "text_content": f"{mem['q']} {mem['r']}",
                "data_type": mem["subtag"],  # 'explicit' or 'implicit'
                "created_timestamp": timestamp
            }
        )
    )

eval_client.upsert(collection_name=EVAL_COLLECTION_NAME, points=points)
print(f"Successfully ingested all {len(points)} memories into Qdrant.")

Generating embeddings and preparing records...
Successfully ingested all 50 memories into Qdrant.


In [16]:
# 5. Execute Formula Query with Time Decay and Payload Boosts
query_text = "Who is my coach?"

# Prepend Nomic's query prefix
query_vector = model.encode(f"search_query: {query_text}").tolist()

current_epoch_time = int(time.time())

# query_points performs the search and modification natively
response = eval_client.query_points(
    collection_name=EVAL_COLLECTION_NAME,
    prefetch=Prefetch(
        query=query_vector,
        limit=50  # Overfetch candidates to run calculation on top matches
    ),
    query=FormulaQuery(
        formula=SumExpression(
            sum=[
                "$score",
                MultExpression(
                    mult=[
                        0.3,
                        FieldCondition(
                            key="data_type",
                            match=MatchValue(value="explicit")
                        )
                    ]
                ),
                ExpDecayExpression(
                    exp_decay=DecayParamsExpression(
                        x="created_timestamp",
                        target=float(current_epoch_time),
                        scale=604800.0,  # 7 days half-life decay
                        midpoint=0.5
                    )
                )
            ]
        )
    ),
    limit=5
)

print(f"=== SEARCH RESULTS FOR: '{query_text}' ===\n")
for point in response.points:
    print(f"ID:              {point.id}")
    print(f"Composite Score: {point.score:.4f}")
    print(f"Payload Text:    {point.payload.get('text_content')}")
    print(f"Type:            {point.payload.get('data_type')}")
    print(f"Age (days):      {(current_epoch_time - point.payload.get('created_timestamp')) / 86400:.1f}")
    print("---")

=== SEARCH RESULTS FOR: 'Who is my coach?' ===

ID:              2
Composite Score: 1.6902
Payload Text:    When was your fitness assessment done? Had a fitness assessment on June 12, 2026. Measured body fat at 18.5 percent.
Type:            explicit
Age (days):      0.6
---
ID:              50
Composite Score: 1.6544
Payload Text:    What did you weigh this morning? Weighed in at 81.8 kilograms this morning. The scale is slowly moving downwards.
Type:            explicit
Age (days):      0.6
---
ID:              4
Composite Score: 1.6105
Payload Text:    What is the dynamic warm-up sequence? Always start with 5 minutes of rowing, followed by 10 leg swings, 10 arm circles, and 5 bodyweight squats.
Type:            explicit
Age (days):      1.9
---
ID:              1
Composite Score: 1.5842
Payload Text:    Who is your personal trainer and when do you meet? My personal trainer is named Marcus, and we meet every Wednesday at 5 PM.
Type:            implicit
Age (days):      0.0
---
ID:   

--- 
## 🚀 Execution of Your Exact Qdrant Client Script

Below is the exact script you provided, configured to query your local Qdrant server (`http://localhost:6333`). 

> **Note on execution**: Set `run_locally_in_memory = True` if you want to test the query logic immediately inside the notebook using Qdrant's in-memory client, or leave it as `False` to send the request directly to your Docker container.

In [17]:
import time
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct,
    Prefetch,
    FormulaQuery,
    SumExpression,
    MultExpression,
    FieldCondition,
    MatchValue,
    ExpDecayExpression,
    DecayParamsExpression
)

# --- CONFIGURATION OPTIONS ---
run_locally_in_memory = True  # Set to True to test instantly in-memory without a running Docker container
COLLECTION_NAME = "agent_memory"
current_epoch_time = int(time.time())
query_vector = [0.15] * 384

if run_locally_in_memory:
    print("Running locally in-memory...")
    client = QdrantClient(":memory:")
    
    # Initialize a 384-dimension collection locally to prevent shape mismatch
    if client.collection_exists(collection_name=COLLECTION_NAME):
        client.delete_collection(collection_name=COLLECTION_NAME)
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=384, distance=Distance.COSINE)
    )
    # Seed a dummy implicit memory to evaluate the boost
    client.upsert(
        collection_name=COLLECTION_NAME,
        points=[
            PointStruct(
                id=999,
                vector=[0.15] * 384,
                payload={
                    "text_content": "Example implicit memory regarding daily hydration targets.",
                    "data_type": "implicit",
                    "created_timestamp": current_epoch_time - 86400  # 1 day old
                }
            )
        ]
    )
else:
    print("Connecting to local Docker container at http://localhost:6333...")
    client = QdrantClient(url="http://localhost:6333")

try:
    # Execute the native payload-based Formula request
    response = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=Prefetch(
            query=query_vector,
            limit=50  # Overfetch candidates to run calculation on top matches
        ),
        query=FormulaQuery(
            formula=SumExpression(
                sum=[
                    "$score",
                    MultExpression(
                        mult=[
                            0.3,
                            FieldCondition(
                                key="data_type",
                                match=MatchValue(value="implicit")
                            )
                        ]
                    ),
                    ExpDecayExpression(
                        exp_decay=DecayParamsExpression(
                            x="created_timestamp",
                            # The ideal value (100% score alignment). Current time = no decay
                            target=float(current_epoch_time),
                            # How fast it drops. E.g., drops to 'midpoint' after 30 days (2,592,000 seconds)
                            scale=2592000.0,
                            # The score output multiplier at the specified scale boundary
                            midpoint=0.5
                        )
                    )
                ]
            )
        ),
        limit=10  # Return final top 10 re-ranked items to user
    )

    # Print execution results sorted by compound math parameters
    print("\n--- Query Execution Successful ---")
    if not response.points:
        print("No points found in the collection. Please ensure your collection is seeded with 384-dimensional vectors.")
    for point in response.points:
        print(f"ID: {point.id}")
        print(f"Composite Score: {point.score}")
        print(f"Payload Text: {point.payload.get('text_content')}")
        print(f"Type: {point.payload.get('data_type')}\n---")
except Exception as e:
    print(f"\nConnection/Query Failed: {e}")
    print("Please verify that Qdrant is running via Docker and the collection exists.")

Running locally in-memory...

--- Query Execution Successful ---
ID: 999
Composite Score: 2.277160167694092
Payload Text: Example implicit memory regarding daily hydration targets.
Type: implicit
---


### 🔬 Does Qdrant solve the Semantic False Positive problem?

While Qdrant **simplifies score fusion** (combining spatial distance, time decay, and metadata filters natively), it **does not solve the core semantic false positive problem** by itself.

#### ⚠️ The Core Problem Remains:
- Dense vector similarity models (bi-encoders) still group conceptually related but functionally different terms together (e.g. mapping `"what exercise am i targetting"` to `"hydration targets"`).
- The database engine's vector search is based strictly on these dense vector distances. Therefore, Qdrant will still return those weak semantic matches in its raw vector search.

#### 💡 Solution:
To achieve 100% precision, we still need a **two-stage hybrid pipeline**:
1. **Stage 1 (Retrieval)**: Use Qdrant with `query_points` (or TF-IDF) to fetch the top 15 candidates quickly.
2. **Stage 2 (Precision)**: Run those 15 candidates through a **Cross-Encoder reranker** or a strict **precision threshold** (`rerank_score >= 0.30`) to block incorrect semantic answers before returning them to the user.